# Примерка на своих фотографиях

Ноутбук берёт свои фото людей и вещей и показывает, как их примеряют обученные модели: базовая (CatVTON) и якорная.

Фото проходят **ровно тот же путь, что обучающие данные**: приведение к канону 768×1024, детекция, разметка одежды, поза, маска области примерки, карты координат и вектор вещи. Это обеспечивает модуль `posefit/tryon.py`, собранный из тех же функций, что датасет обучения. Генерация общая с `scripts/evaluate.py`: модель собирается по снимку своего прогона, стартовый шум для пары фиксирован.

**Где запускать:** на сервере с GPU, в репозитории на ветке `feature/anchor`. Ноутбук сам перейдёт в корень репозитория, если лежит в `notebooks/`.

**Как подготовить фото:**

* **человек** — один человек в кадре, торс виден целиком (плечи и бёдра), лучше анфас; кадр по пояс подходит для верха;
* **вещь** — плоская выкладка на светлом однотонном фоне, вещь целиком, как в карточке товара;
* **группа одежды** задаёт, какую область на человеке перерисовывать:
  `upper` — верх, `lower` — брюки, юбки, шорты, `dress` — платья, `outer` — верхняя одежда.

Свои фото удобно класть в `my_examples/` — каталог не попадает в git.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "posefit").exists():
    ROOT = ROOT.parent          # ноутбук лежит в notebooks/
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from posefit.tryon import GROUPS, Preprocessor, TryOn, build_item, composite, uncrop

print("корень репозитория:", ROOT)
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| карта видна:", torch.cuda.is_available())

## Параметры

Здесь всё, что меняется от запуска к запуску. Модели задаются каталогами прогонов: схема, маска и разрешение каждой берутся из её снимка `train_config.yaml`, как при замере.

In [ ]:
# Модели для сравнения: подпись -> каталог прогона.
RUNS = {
    "базовая": "runs/studio_wild",
    "якорная": "runs/studio_wild_anchor",
}
CHECKPOINT = "final.pt"
ZERO_SHOT = False            # добавить исходный SD inpainting без обучения — контроль
TRAIN_CONFIG = "configs/train.yaml"
DEVICE = "cuda:1"

# Пары для примерки: (фото человека, фото вещи, группа одежды).
PAIRS = [
    ("my_examples/person_1.jpg", "my_examples/garment_1.jpg", "upper"),
]
# Все люди на все вещи одной группы:
# persons = sorted(Path("my_examples/persons").glob("*.jpg"))
# garments = sorted(Path("my_examples/garments").glob("*.jpg"))
# PAIRS = [(p, g, "upper") for p in persons for g in garments]

STEPS = 50                   # шагов DDIM, как при замере
SEED = 0                     # база стартового шума; при том же SEED результат повторяется
BATCH_SIZE = 4
TOKEN = "studio"             # токен надёжности на выводе (только якорная схема)
GUIDANCE = 0.0               # guidance по надёжности: 0 — выключен, 2 — как в замере @g2
PASTE_BACK = True            # вне маски вернуть исходные пиксели фото
UNCROP = True                # убрать белые поля, вернуть пропорции исходного фото
OUT_DIR = Path("my_examples/results")

## Проверка входов

Файлы проверяются до загрузки тяжёлых моделей: опечатка в пути не должна стоить минуты ожидания.

In [ ]:
problems = []
for person_path, garment_path, group in PAIRS:
    for path in (person_path, garment_path):
        if not Path(path).exists():
            problems.append(f"нет файла: {path}")
    if group not in GROUPS:
        problems.append(f"группа {group!r} для {person_path}: ожидается одна из {GROUPS}")
for name, run in RUNS.items():
    if not (Path(run) / CHECKPOINT).exists():
        problems.append(f"модель «{name}»: нет {Path(run) / CHECKPOINT}")
if problems:
    raise FileNotFoundError("\n".join(problems))
print(f"пар: {len(PAIRS)}, моделей: {len(RUNS) + int(ZERO_SHOT)}")

## Препроцессинг

Детекция, разметка, поза и вектор вещи — те же модели и настройки, что в `scripts/preprocess.py`. Предупреждения стоит прочитать, при них результат будет хуже:

* **торс не виден** — координаты строятся по маске, а не по позе;
* **вещь на выкладке не распознана** — рамка вещи берётся по не-белым пикселям;
* **одежды группы мало** — чаще всего разметка отнесла однотонный образ к «платью», и маска группы `upper` закроет только рукава. Помогает другое фото или группа `dress`.

In [ ]:
pre = Preprocessor(device=DEVICE)
pairs = []
for person_path, garment_path, group in PAIRS:
    pair_id = f"{Path(person_path).stem}__{Path(garment_path).stem}"
    person = pre.person(person_path, group)
    garment = pre.garment(garment_path, group)
    pairs.append({"id": pair_id, "group": group, "person": person, "garment": garment})
    print(f"{pair_id}: людей в кадре {person.n_persons}")
    for warning in person.warnings + garment.warnings:
        print("   !", warning)

Контактный лист: что увидит модель. Слева направо — человек с маской области примерки, вещь, карта u и карта v в системе торса. У человека и вещи одна координата должна давать один градиент: плечо футболки там же по цвету, где плечо человека.

In [ ]:
def heat(values):
    red = ((values + 1.0) / 2.0 * 255).clip(0, 255).astype(np.uint8)
    return np.stack([red, np.zeros_like(red), 255 - red], axis=-1)

def to_u8(chw):
    return ((chw.transpose(1, 2, 0) + 1.0) * 127.5).clip(0, 255).astype(np.uint8)

# Карты u и v склеены из двух половин, поэтому им вдвое больше ширины.
fig, axes = plt.subplots(len(pairs), 4, figsize=(15, 4.2 * len(pairs)), squeeze=False,
                         gridspec_kw={"width_ratios": [1, 1, 2, 2]})
for row, pair in zip(axes, pairs):
    # Только для показа: рабочее разрешение 384x512, подсказки включены.
    item = build_item(pair["person"], pair["garment"], pair["group"], pair["id"],
                      flags={"guide": True})
    person, inside = to_u8(item["person"]), item["mask"][0][..., None]
    overlay = (person * (1 - 0.45 * inside) + np.array([255, 60, 60]) * 0.45 * inside).astype(np.uint8)
    skeleton = item["guide_person"][2] > 0.3
    overlay[skeleton] = (255, 255, 0)
    panels = [overlay, to_u8(item["garment"]),
              heat(np.concatenate([item["guide_person"][0], item["guide_garment"][0]], axis=1)),
              heat(np.concatenate([item["guide_person"][1], item["guide_garment"][1]], axis=1))]
    titles = ["человек, маска, скелет", "вещь", "u: человек | вещь", "v: человек | вещь"]
    for ax, image, title in zip(row, panels, titles):
        ax.imshow(image); ax.set_title(title, fontsize=9); ax.axis("off")
    row[0].set_ylabel(pair["id"])
plt.tight_layout(); plt.show()

## Модели

Каждая модель собирается по снимку своего прогона: схема, маска и разрешение оттуда, как при замере. Две модели SD 1.5 в fp16 занимают около 4 ГБ видеопамяти.

In [ ]:
models = {}
for name, run in RUNS.items():
    models[name] = TryOn(run, CHECKPOINT, device=DEVICE, train_config=TRAIN_CONFIG)
if ZERO_SHOT:
    models["без обучения"] = TryOn(None, "none", device=DEVICE, train_config=TRAIN_CONFIG)
for name, model in models.items():
    print(f"{name:14} схема {model.flags['name']:8} шаг {model.step:6}  маска {model.mask_stage:17} "
          f"{model.size[0]}x{model.size[1]}  ({model.source})")

## Примерка

Стартовый шум зависит от пары и `SEED`, но не от модели: разница между столбцами — разница моделей, а не случайного шума.

In [ ]:
def finish(result, item, original_size):
    image = composite(result, item) if PASTE_BACK else result
    return uncrop(image, original_size) if UNCROP else Image.fromarray(image)

results = {pair["id"]: {} for pair in pairs}
for name, model in models.items():
    for start in range(0, len(pairs), BATCH_SIZE):
        chunk = pairs[start:start + BATCH_SIZE]
        items = [model.item(p["person"], p["garment"], p["group"], p["id"]) for p in chunk]
        token, guidance = (TOKEN, GUIDANCE) if model.flags["reliability"] else ("studio", 0.0)
        outputs = model(items, steps=STEPS, seed=SEED, token=token, guidance=guidance)
        for pair, item, output in zip(chunk, items, outputs):
            results[pair["id"]][name] = finish(output, item, pair["person"].original_size)
    print("готово:", name)

In [ ]:
columns = ["человек", "вещь", *models]
fig, axes = plt.subplots(len(pairs), len(columns), figsize=(3.2 * len(columns), 4.4 * len(pairs)),
                         squeeze=False)
for row, pair in zip(axes, pairs):
    person = uncrop(np.asarray(pair["person"].canonical), pair["person"].original_size)
    garment = uncrop(np.asarray(pair["garment"].canonical), pair["garment"].original_size)
    for ax, image, title in zip(row, [person, garment, *results[pair["id"]].values()], columns):
        ax.imshow(image); ax.set_title(title, fontsize=10); ax.axis("off")
plt.tight_layout()
OUT_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT_DIR / "comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
for pair_id, by_model in results.items():
    for name, image in by_model.items():
        image.save(OUT_DIR / f"{pair_id}__{name.replace(' ', '_')}.png")
print("сохранено в", OUT_DIR.resolve())

## Надёжность пары: токен и guidance

Только для якорной схемы. Первая пара генерируется с разной силой guidance по надёжности и с токеном шумной пары вместо студийного. Если токен ничего не меняет, модель им не пользуется.

In [ ]:
anchor = next((m for m in models.values() if m.flags["reliability"]), None)
if anchor is None:
    print("среди моделей нет якорной с токеном надёжности")
else:
    pair = pairs[0]
    item = anchor.item(pair["person"], pair["garment"], pair["group"], pair["id"])
    variants = [("studio", 0.0), ("studio", 1.5), ("studio", 2.0), ("studio", 3.0), ("wild_noisy", 0.0)]
    fig, axes = plt.subplots(1, len(variants), figsize=(3.2 * len(variants), 4.4))
    for ax, (token, guidance) in zip(axes, variants):
        output = anchor([item], steps=STEPS, seed=SEED, token=token, guidance=guidance)[0]
        ax.imshow(finish(output, item, pair["person"].original_size))
        ax.set_title(f"{token}, guidance {guidance:g}", fontsize=9); ax.axis("off")
    plt.tight_layout(); plt.show()